# Lesson 8 — Modules and ppx_deriving_yojson

**Read `lesson_8.ml` first.**
This notebook is the RUN environment for the ppx parts (Parts 9-12).
Parts 1-8 (pure modules) run on ocaml.org/play with no setup needed.

Run cells **top to bottom in order.**

## CELL 1: Install OCaml, Yojson, and ppx_deriving_yojson

Run once per session. Takes ~3 minutes.

In [ ]:
!apt-get install -qq -y ocaml ocaml-nox ocaml-findlib opam
!opam init --disable-sandboxing -y -q
!eval $(opam env) && opam install -y -q yojson
!eval $(opam env) && opam install -y ocamlfind
!eval $(opam env) && opam install -y -q ppx_deriving_yojson
print('Done.')

## CELL 2: Confirm install

In [ ]:
!eval $(opam env) && ocamlfind list | grep -E 'yojson|ppx_deriving'

You should see lines for both `yojson` and `ppx_deriving_yojson`.

---
## CELL 3: The validator BEFORE ppx (lesson 7 style)

This is the old version — 80+ lines of hand-written parsing.
We compile and run it first so you can compare.

In [ ]:
before_ppx = r"""
type payload = {
  user_id     : string;
  user_name   : string;
  query       : string;
  ssn         : string option;
  credit_card : string option;
}

type validation_error =
  | PiiFieldPresent      of { field_name : string }
  | RequiredFieldMissing of { field_name : string }
  | FieldTooLong         of { field_name : string; max_len : int; actual_len : int }

(* HAND-WRITTEN PARSING -- 40+ lines *)
let parse_string_field key fields =
  match List.assoc_opt key fields with
  | Some (`String s) -> Ok s
  | Some _           -> Error ("field '" ^ key ^ "' has wrong type")
  | None             -> Error ("field '" ^ key ^ "' is missing")

let parse_optional_string_field key fields =
  match List.assoc_opt key fields with
  | None             -> Ok None
  | Some `Null       -> Ok None
  | Some (`String s) -> Ok (Some s)
  | Some _           -> Error ("field '" ^ key ^ "' has wrong type")

let parse_payload json_str =
  let json =
    try Ok (Yojson.Safe.from_string json_str)
    with _ -> Error "invalid JSON"
  in
  match json with
  | Error e -> Error e
  | Ok (`Assoc fields) ->
      (match parse_string_field "user_id" fields with
      | Error e -> Error e
      | Ok user_id ->
        match parse_string_field "user_name" fields with
        | Error e -> Error e
        | Ok user_name ->
          match parse_string_field "query" fields with
          | Error e -> Error e
          | Ok query ->
            match parse_optional_string_field "ssn" fields with
            | Error e -> Error e
            | Ok ssn ->
              match parse_optional_string_field "credit_card" fields with
              | Error e -> Error e
              | Ok credit_card ->
                Ok { user_id; user_name; query; ssn; credit_card })
  | Ok _ -> Error "expected a JSON object"

let check_no_ssn p =
  match p.ssn with
  | None   -> Ok p
  | Some _ -> Error (PiiFieldPresent { field_name = "ssn" })

let check_no_credit_card p =
  match p.credit_card with
  | None   -> Ok p
  | Some _ -> Error (PiiFieldPresent { field_name = "credit_card" })

let check_query_not_empty p =
  if String.length p.query = 0
  then Error (RequiredFieldMissing { field_name = "query" })
  else Ok p

let validate_payload p =
  match check_no_ssn p with
  | Error e -> Error e
  | Ok p ->
    match check_no_credit_card p with
    | Error e -> Error e
    | Ok p -> check_query_not_empty p

let describe_error err =
  match err with
  | PiiFieldPresent { field_name } ->
      Printf.sprintf "PII field '%s' must not be present" field_name
  | RequiredFieldMissing { field_name } ->
      Printf.sprintf "Required field '%s' is missing or empty" field_name
  | FieldTooLong { field_name; max_len; actual_len } ->
      Printf.sprintf "Field '%s' is too long: max %d, got %d" field_name max_len actual_len

let payload_to_json p =
  `Assoc [
    ("user_id",     `String p.user_id);
    ("user_name",   `String p.user_name);
    ("query",       `String p.query);
    ("ssn",         (match p.ssn with None -> `Null | Some s -> `String s));
    ("credit_card", (match p.credit_card with None -> `Null | Some s -> `String s));
  ]

let make_ok_response p =
  `Assoc [ ("status", `String "ok"); ("payload", payload_to_json p) ]

let make_error_response msg =
  `Assoc [ ("status", `String "error"); ("message", `String msg) ]

let () =
  let input =
    let buf = Buffer.create 256 in
    (try while true do Buffer.add_channel buf stdin 1 done
     with End_of_file -> ());
    Buffer.contents buf
  in
  match parse_payload input with
  | Error e ->
      print_string (Yojson.Safe.to_string (make_error_response ("parse error: " ^ e)));
      exit 1
  | Ok payload ->
    match validate_payload payload with
    | Error e ->
        print_string (Yojson.Safe.to_string (make_error_response (describe_error e)));
        exit 1
    | Ok clean ->
        print_string (Yojson.Safe.to_string (make_ok_response clean));
        exit 0
"""

with open('validator_before.ml', 'w') as f:
    f.write(before_ppx)

print(f'Written: {len(before_ppx.splitlines())} lines')

## CELL 4: The validator AFTER ppx

Same behaviour. Much less code. Count the lines.

In [ ]:
after_ppx = r"""
(* [@@deriving yojson] generates payload_of_yojson and payload_to_yojson *)
type payload = {
  user_id     : string;
  user_name   : string;
  query       : string;
  ssn         : string option;
  credit_card : string option;
} [@@deriving yojson]

type validation_error =
  | PiiFieldPresent      of { field_name : string }
  | RequiredFieldMissing of { field_name : string }
  | FieldTooLong         of { field_name : string; max_len : int; actual_len : int }

(* PARSING -- 4 lines instead of 40 *)
let parse_payload json_str =
  try
    let json = Yojson.Safe.from_string json_str in
    match payload_of_yojson json with
    | Ok p    -> Ok p
    | Error e -> Error ("parse error: " ^ e)
  with _ -> Error "parse error: invalid JSON"

(* VALIDATORS -- identical to lesson 7 *)
let check_no_ssn p =
  match p.ssn with
  | None   -> Ok p
  | Some _ -> Error (PiiFieldPresent { field_name = "ssn" })

let check_no_credit_card p =
  match p.credit_card with
  | None   -> Ok p
  | Some _ -> Error (PiiFieldPresent { field_name = "credit_card" })

let check_query_not_empty p =
  if String.length p.query = 0
  then Error (RequiredFieldMissing { field_name = "query" })
  else Ok p

let check_query_length p =
  let max_len = 500 in
  let actual  = String.length p.query in
  if actual > max_len
  then Error (FieldTooLong { field_name = "query"; max_len; actual_len = actual })
  else Ok p

let validate_payload p =
  match check_no_ssn p with
  | Error e -> Error e
  | Ok p ->
    match check_no_credit_card p with
    | Error e -> Error e
    | Ok p ->
      match check_query_not_empty p with
      | Error e -> Error e
      | Ok p -> check_query_length p

let describe_error err =
  match err with
  | PiiFieldPresent { field_name } ->
      Printf.sprintf "PII field '%s' must not be present" field_name
  | RequiredFieldMissing { field_name } ->
      Printf.sprintf "Required field '%s' is missing or empty" field_name
  | FieldTooLong { field_name; max_len; actual_len } ->
      Printf.sprintf "Field '%s' is too long: max %d, got %d"
        field_name max_len actual_len

(* OUTPUT -- payload_to_yojson is ppx-generated *)
let make_ok_response p =
  `Assoc [ ("status", `String "ok"); ("payload", payload_to_yojson p) ]

let make_error_response msg =
  `Assoc [ ("status", `String "error"); ("message", `String msg) ]

let () =
  let input =
    let buf = Buffer.create 256 in
    (try while true do Buffer.add_channel buf stdin 1 done
     with End_of_file -> ());
    Buffer.contents buf
  in
  match parse_payload input with
  | Error e ->
      print_string (Yojson.Safe.to_string (make_error_response e));
      exit 1
  | Ok payload ->
    match validate_payload payload with
    | Error e ->
        print_string (Yojson.Safe.to_string (make_error_response (describe_error e)));
        exit 1
    | Ok clean ->
        print_string (Yojson.Safe.to_string (make_ok_response clean));
        exit 0
"""

with open('validator_after.ml', 'w') as f:
    f.write(after_ppx)

print(f'Written: {len(after_ppx.splitlines())} lines')

## CELL 5: Compile both versions

Notice the compile command for ppx — it adds `-package ppx_deriving_yojson`.

In [ ]:
import subprocess

# Compile BEFORE version (lesson 7 style -- no ppx)
r1 = subprocess.run(
    'eval $(opam env) && ocamlfind ocamlopt -package yojson -linkpkg validator_before.ml -o validator_before',
    shell=True, capture_output=True, text=True, executable='/bin/bash'
)
print('BEFORE:', 'OK' if r1.returncode == 0 else 'FAILED')
if r1.stderr.strip(): print(r1.stderr[:300])

# Compile AFTER version (with ppx -- note the extra -package)
r2 = subprocess.run(
    'eval $(opam env) && ocamlfind ocamlopt -package yojson,ppx_deriving_yojson -linkpkg validator_after.ml -o validator_after',
    shell=True, capture_output=True, text=True, executable='/bin/bash'
)
print('AFTER: ', 'OK' if r2.returncode == 0 else 'FAILED')
if r2.stderr.strip(): print(r2.stderr[:300])

Both should print OK.

**KEY DIFFERENCE in the compile command:**
- Before: `-package yojson`
- After:  `-package yojson,ppx_deriving_yojson`

The ppx package is added as a preprocessor.
It runs BEFORE the compiler and generates the parsing functions.
The compiler then compiles the generated code as if you wrote it.

---
## CELL 6: Run the same tests on both versions

Output should be IDENTICAL — same behaviour, less code.

In [ ]:
import subprocess, json

test_cases = [
    {"user_id": "u_1", "user_name": "Alice", "query": "What is the weather?", "ssn": None,          "credit_card": None},
    {"user_id": "u_2", "user_name": "Bob",   "query": "Help me",              "ssn": "123-45-6789", "credit_card": None},
    {"user_id": "u_3", "user_name": "Carol", "query": "",                     "ssn": None,          "credit_card": None},
    {                   "user_name": "Dave",  "query": "Hello",                "ssn": None,          "credit_card": None},
    {"user_id": "u_5", "user_name": "Eve",   "query": "Buy something",        "ssn": None,          "credit_card": "4111111111111111"},
]

def run_validator(binary, payload):
    result = subprocess.run(
        [binary],
        input=json.dumps(payload),
        capture_output=True, text=True
    )
    response = json.loads(result.stdout)
    uid = payload.get('user_id', 'NO_ID')
    if response['status'] == 'ok':
        return f'[{uid}] PASSED'
    else:
        return f'[{uid}] FAILED: {response["message"]}'

print('=== BEFORE (lesson 7 style, hand-written parsing) ===')
for p in test_cases:
    print(run_validator('./validator_before', p))

print()
print('=== AFTER (ppx_deriving_yojson, generated parsing) ===')
for p in test_cases:
    print(run_validator('./validator_after', p))

Both sections should print the exact same output:
```
[u_1]   PASSED
[u_2]   FAILED: PII field 'ssn' must not be present
[u_3]   FAILED: Required field 'query' is missing or empty
[NO_ID] FAILED: parse error: ...
[u_5]   FAILED: PII field 'credit_card' must not be present
```

**Same behaviour. The AFTER version just has ~40 fewer lines of boilerplate.**

---
## CELL 7: See ppx in action -- add a new field

This demonstrates why ppx saves time.
Add a new field `user_role` to the payload.

With hand-written parsing (lesson 7 style):
you would add a new `parse_string_field "user_role"` match arm.

With ppx: just add the field to the type. Done.

In [ ]:
extended = r"""
(* Added user_role field -- ppx handles the parsing automatically *)
type payload = {
  user_id     : string;
  user_name   : string;
  query       : string;
  ssn         : string option;
  credit_card : string option;
  user_role   : string option;  (* NEW FIELD -- no other changes needed *)
} [@@deriving yojson]

type validation_error =
  | PiiFieldPresent      of { field_name : string }
  | RequiredFieldMissing of { field_name : string }
  | FieldTooLong         of { field_name : string; max_len : int; actual_len : int }

let parse_payload json_str =
  try
    let json = Yojson.Safe.from_string json_str in
    match payload_of_yojson json with
    | Ok p    -> Ok p
    | Error e -> Error ("parse error: " ^ e)
  with _ -> Error "parse error: invalid JSON"

let check_no_ssn p =
  match p.ssn with
  | None   -> Ok p
  | Some _ -> Error (PiiFieldPresent { field_name = "ssn" })

let check_no_credit_card p =
  match p.credit_card with
  | None   -> Ok p
  | Some _ -> Error (PiiFieldPresent { field_name = "credit_card" })

let check_query_not_empty p =
  if String.length p.query = 0
  then Error (RequiredFieldMissing { field_name = "query" })
  else Ok p

let validate_payload p =
  match check_no_ssn p with
  | Error e -> Error e
  | Ok p ->
    match check_no_credit_card p with
    | Error e -> Error e
    | Ok p -> check_query_not_empty p

let describe_error err =
  match err with
  | PiiFieldPresent { field_name } ->
      Printf.sprintf "PII field '%s' must not be present" field_name
  | RequiredFieldMissing { field_name } ->
      Printf.sprintf "Required field '%s' is missing or empty" field_name
  | FieldTooLong { field_name; max_len; actual_len } ->
      Printf.sprintf "Field '%s' is too long: max %d, got %d" field_name max_len actual_len

let make_ok_response p =
  `Assoc [ ("status", `String "ok"); ("payload", payload_to_yojson p) ]

let make_error_response msg =
  `Assoc [ ("status", `String "error"); ("message", `String msg) ]

let () =
  let input =
    let buf = Buffer.create 256 in
    (try while true do Buffer.add_channel buf stdin 1 done
     with End_of_file -> ());
    Buffer.contents buf
  in
  match parse_payload input with
  | Error e ->
      print_string (Yojson.Safe.to_string (make_error_response e));
      exit 1
  | Ok payload ->
    match validate_payload payload with
    | Error e ->
        print_string (Yojson.Safe.to_string (make_error_response (describe_error e)));
        exit 1
    | Ok clean ->
        print_string (Yojson.Safe.to_string (make_ok_response clean));
        exit 0
"""

with open('validator_extended.ml', 'w') as f:
    f.write(extended)

import subprocess
r = subprocess.run(
    'eval $(opam env) && ocamlfind ocamlopt -package yojson,ppx_deriving_yojson -linkpkg validator_extended.ml -o validator_extended',
    shell=True, capture_output=True, text=True, executable='/bin/bash'
)
print('Compile:', 'OK' if r.returncode == 0 else 'FAILED')
if r.stderr.strip(): print(r.stderr[:300])

# Test with the new field included
import json
payload = {
    "user_id": "u_1",
    "user_name": "Alice",
    "query": "What is the weather?",
    "ssn": None,
    "credit_card": None,
    "user_role": "admin"  # new field
}
result = subprocess.run(['./validator_extended'], input=json.dumps(payload), capture_output=True, text=True)
response = json.loads(result.stdout)
print('Result:', response)

The new `user_role` field was parsed automatically by ppx.
No changes to parse_payload, no new match arms, nothing.
Just add the field to the type.

---
## Summary

| | Lesson 7 (hand-written) | Lesson 8 (ppx) |
|---|---|---|
| Parsing code | ~40 lines | 4 lines |
| Output code | ~10 lines | 1 line (`payload_to_yojson`) |
| Add a field | Update parse_payload manually | Add to type, done |
| Behaviour | Identical | Identical |
| Compile command | `-package yojson` | `-package yojson,ppx_deriving_yojson` |

ppx is a preprocessor — it runs before the compiler and generates code from annotations.
The compiled binary is identical. The source code is much shorter.